# Descriptives: Patent Quality Indicators

## Purpose
Compute the patent-level quality indicators (FAMSIZE, BCIT, FCIT, REFNPAT) used
in the Dieselgate triple-difference analysis, and produce the density plots.

## Sample
- **Firms**: 18 brand groups (3 German treatment + 15 control)
- **Years**: the checkpoint covers 2010-2019
- **Patents**: US applications (appln_auth == 'US'), CPC-filtered to ICE +
  EV/Hybrid technology classes, continuations removed

## Sections
1. Imports and constants
2. Build the focal sample
3. FAMSIZE - family size
4. Citation indicators - BCIT, FCIT, REFNPAT
5. Density plots

## 1. Imports and Constants

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
import gc
import random
from pathlib import Path

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi'        : 150,
    'font.size'         : 11,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
})


In [ ]:
# ── File paths ────────────────────────────────────────────────────────────────
# All paths are relative to 02_quality_indicators/ (the notebook's working directory).

# Produced by step4.
US_PATENTS_FILE = Path('../01_patstat_pipeline/step4/us_patents_clean.csv')

# step3 output — too large to copy (4.6 GB).
CPC_FILE = Path('../01_patstat_pipeline/step3/patents_cpc_classified.csv')

# Raw PATSTAT citation tables — too large to copy (~9 GB and ~20 GB).
# They live outside the repository (see SETUP.md, Part 2). Default: a folder
# `patstat_raw/` next to the repository folder; set the environment variable
# PATSTAT_DIR to use another location.
import os
PATSTAT_DIR  = Path(os.environ.get('PATSTAT_DIR', '../../patstat_raw'))
TLS211_PARTS = [PATSTAT_DIR / 'tls211_part01.csv', PATSTAT_DIR / 'tls211_part02.csv']
TLS212_PARTS = [PATSTAT_DIR / 'tls212_part01.csv',
                PATSTAT_DIR / 'tls212_part02.csv',
                PATSTAT_DIR / 'tls212_part03.csv']

# Output directories
OUTPUT_DIR   = Path('output')
PLOTS_RAW    = OUTPUT_DIR / 'plots/raw_density'
TABLES_DIR   = OUTPUT_DIR / 'tables'

# Checkpoints — the PATSTAT extracts the indicators are built from
CHECKPOINTS_DIR = Path('data/checkpoints')

for d in [PLOTS_RAW, TABLES_DIR, CHECKPOINTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Analysis parameters ───────────────────────────────────────────────────────
YEAR_START  = 2010
YEAR_END    = 2021      # checkpoint window. The analysis sample is 2010-2019 —
                        # every downstream script filters to it, because forward
                        # citations for 2020-2021 filings are still truncated
                        # (mean FCIT: 0.25 in 2019, 0.04 in 2020, 0.00 in 2021).
CHUNK_SIZE  = 2_000_000  # rows per chunk for large-file reading

# All inputs must be present before running: see SETUP.md for where to place the
# raw PATSTAT tables. The scans below write their results to data/checkpoints/.
print('Inputs:')
missing = []
for p in [US_PATENTS_FILE, CPC_FILE] + TLS211_PARTS + TLS212_PARTS:
    ok = p.exists()
    if not ok:
        missing.append(p)
    print(f"  {'ok     ' if ok else 'MISSING'}  {p}")
if missing:
    raise FileNotFoundError(
        f'{len(missing)} input file(s) missing. Raw PATSTAT exports (TLS211/212) belong in '
        f'{PATSTAT_DIR.resolve()} (default: patstat_raw/ next to the repository folder); '
        'files from 01_patstat_pipeline come from running its steps. See SETUP.md, Part 2.')


In [ ]:
# ── COMPANY_MAP ──────────────────────────────────────────────────────────────
# Maps PATSTAT PSN_NAMEs to consolidated brand labels.
# Restricted to the 57 psn_names that survive the 2010–2020 time-window and
# ICE/EV relevance filter.  Entries absent from that filtered set are excluded.
COMPANY_MAP = {
    # ── German treatment group ────────────────────────────────────────────────
    'VOLKSWAGEN'                     : 'Volkswagen Group',
    'AUDI'                           : 'Volkswagen Group',
    'PORSCHE'                        : 'Volkswagen Group',
    'DAIMLER'                        : 'Daimler',
    'BMW (BAYERISCHE MOTOREN WERKE)' : 'BMW',
    # ── American ─────────────────────────────────────────────────────────────
    'FORD GLOBAL TECHNOLOGIES'        : 'Ford',
    'FORD MOTOR COMPANY'              : 'Ford',
    'FORD-WERKE'                      : 'Ford',
    'LINCOLN GLOBAL'                  : 'Ford',
    'LINCOLN'                         : 'Ford',
    'GM (GENERAL MOTORS CORPORATION)' : 'General Motors',
    # GM files under several legal entities; 'GM GLOBAL TECHNOLOGY OPERATIONS'
    # holds the bulk of the modern portfolio, the legacy name only ~20 apps.
    'GM GLOBAL TECHNOLOGY OPERATIONS'     : 'General Motors',
    'GENERAL MOTORS RESEARCH CORPORATION' : 'General Motors',
    'SAIC GENERAL MOTORS CORPORATION'     : 'General Motors',
    'SHANGHAI GENERAL MOTORS COMPANY'     : 'General Motors',
    'SAIC-GM-WULING AUTOMOBILE COMPANY'   : 'General Motors',
    # ── Japanese ─────────────────────────────────────────────────────────────
    'TOYOTA MOTOR CORPORATION'                               : 'Toyota',
    'TOYOTA INDUSTRIES CORPORATION'                          : 'Toyota',
    'TOYOTA MOTOR ENGINEERING & MANUFACTURING NORTH AMERICA' : 'Toyota',
    'TOYOTA BOSHOKU CORPORATION'                             : 'Toyota',
    'TOYOTA CHUO KENKYUSHO'                                  : 'Toyota',
    'TOYOTA SHATAI'                                          : 'Toyota',
    'TOYOTA MOTOR EUROPE'                                    : 'Toyota',
    'TOYOTA INFOTECHNOLOGY CENTER COMPANY'                   : 'Toyota',
    'TOYOTA CENTRAL RES & DEV LAB'                           : 'Toyota',
    'TOYOTA MOTOR EAST JAPAN'                                : 'Toyota',
    'SUZUKI MOTOR CORPORATION'                               : 'Suzuki',
    'HONDA MOTOR COMPANY'                                    : 'Honda',
    'HONDA ACCESS CORPORATION'                               : 'Honda',
    'HONDA LOCK MANUFACTURING COMPANY'                       : 'Honda',
    'HONDA LOCK'                                             : 'Honda',
    'NISSAN MOTOR COMPANY'                                   : 'Nissan',
    'NISSAN NORTH AMERICA'                                   : 'Nissan',
    'MITSUBISHI MOTORS CORPORATION'                          : 'Mitsubishi',
    'MITSUBISHI HEAVY INDUSTRIES ENGINE & TURBOCHARGER'      : 'Mitsubishi',
    'MITSUBISHI HEAVY INDUSTRIES THERMAL SYSTEMS COMPANY'    : 'Mitsubishi',
    'MITSUBISHI FUSO TRUCK AND BUS CORPORATION'              : 'Mitsubishi',
    'MITSUBISHI JIDOSHA ENGINEERING'                         : 'Mitsubishi',
    'MITSUBISHI HEAVY INDUSTRIES ENGINEERING'                : 'Mitsubishi',
    'MITSUBISHI HEAVY INDUSTRIES AUTOMOTIVE THERMAL SYSTEMS COMPANY' : 'Mitsubishi',
    'MAZDA MOTOR CORPORATION'                                : 'Mazda',
    'SUBARU'                                                 : 'Subaru',
    # ── Korean ───────────────────────────────────────────────────────────────
    'HYUNDAI MOTOR COMPANY'      : 'Hyundai-Kia',
    'HYUNDAI MOBIS COMPANY'      : 'Hyundai-Kia',
    'HYUNDAI AUTRON COMPANY'     : 'Hyundai-Kia',
    'HYUNDAI DYMOS'              : 'Hyundai-Kia',
    'HYUNDAI KEFICO CORPORATION' : 'Hyundai-Kia',
    'HYUNDAI WIA CORPORATION'    : 'Hyundai-Kia',
    'HYUNDAI TRANSYS'            : 'Hyundai-Kia',
    'KIA MOTORS CORPORATION'     : 'Hyundai-Kia',
    'KIA CORPORATION'            : 'Hyundai-Kia',
    # ── Swedish / British ─────────────────────────────────────────────────────
    'VOLVO CAR CORPORATION'       : 'Volvo Cars',
    'VOLVO TECHNOLOGY CORPORATION': 'Volvo Cars',
    'VOLVO LASTVAGNAR'            : 'Volvo Trucks',
    'VOLVO TRUCK CORPORATION'     : 'Volvo Trucks',
    'AB VOLVO PENTA'              : 'Volvo Trucks',
    'JAGUAR LAND ROVER'           : 'Jaguar Land Rover',
    'JAGUAR CARS'                 : 'Jaguar Land Rover',
    # ── Italian ──────────────────────────────────────────────────────────────
    'FIAT AUTO'                   : 'Fiat',
    'CRF (CENTRO RICERCHE FIAT)'  : 'Fiat',
    # ── Other ─────────────────────────────────────────────────────────────────
    'TESLA'       : 'Tesla',
    'TESLA MOTORS': 'Tesla',
}

# Final brand order — 18 groups
BRAND_ORDER = [
    'Volkswagen Group', 'Daimler', 'BMW',           # treatment
    'Ford', 'General Motors',                        # American
    'Toyota', 'Honda', 'Nissan', 'Mitsubishi',
    'Mazda', 'Suzuki', 'Subaru',                    # Japanese
    'Hyundai-Kia',                                   # Korean
    'Volvo Cars', 'Volvo Trucks',                    # Swedish
    'Jaguar Land Rover',                             # British
    'Fiat',                                          # Italian
    'Tesla',                                         # Other
]

GERMAN_BRANDS = ['Volkswagen Group', 'Daimler', 'BMW']

print(f'COMPANY_MAP covers {len(COMPANY_MAP)} PSN_NAMEs → {len(set(COMPANY_MAP.values()))} brand groups')
print(f'BRAND_ORDER: {len(BRAND_ORDER)} brands')


## 2. Build the focal sample

Construct the focal sample by reading the step4 output (`us_patents_clean.csv`),
which already has:
- `appln_auth == 'US'` filter applied
- `appln_id <= 900 000 000` (artificial records removed)
- Continuation patents excluded via tls216

Additional filters applied here:
- Year: 2010–2021 (checkpoint window; analysis sample is 2010–2019)
- CPC relevance: ≥1 ICE or EV/Hybrid code
- Brand mapping via COMPANY_MAP (18 final brands)
- Deduplication: one row per appln_id (a patent filed by multiple subsidiaries
  of the same brand produces one row after mapping)


In [ ]:
def build_df_focal(filepath, year_start, year_end, company_map, brand_order):
    """
    Build the focal sample from the step4 output (us_patents_clean.csv).

    Applies year filter, CPC relevance filter, brand mapping, and deduplication.
    Does NOT re-apply appln_auth or continuation filters — those are already
    in us_patents_clean.csv.

    Parameters
    ----------
    filepath   : Path — us_patents_clean.csv
    year_start : int
    year_end   : int
    company_map : dict
    brand_order : list of str — only these 18 brands are retained

    Returns
    -------
    pd.DataFrame with columns:
        appln_id, docdb_family_id, year, brand, tech_type
    where tech_type ∈ {'ICE', 'EV_Hybrid'}
    """
    print(f'Loading {filepath.name}...')
    df = pd.read_csv(
        filepath,
        usecols=['appln_id', 'docdb_family_id', 'appln_filing_date', 'psn_name',
                 'count_ICE_codes', 'count_EV_codes', 'count_Hybrid_codes'],
        dtype={
            'appln_id'          : 'int32',
            'docdb_family_id'   : 'int32',
            'count_ICE_codes'   : 'int16',
            'count_EV_codes'    : 'int16',
            'count_Hybrid_codes': 'int16',
        },
        parse_dates=['appln_filing_date'],
    )
    print(f'  Loaded: {len(df):,} rows')

    # Year filter
    df['year'] = df['appln_filing_date'].dt.year
    df = df[(df['year'] >= year_start) & (df['year'] <= year_end)].copy()
    print(f'  After year filter ({year_start}–{year_end}): {len(df):,} rows')

    # CPC relevance filter
    df = df[
        (df['count_ICE_codes']   > 0) |
        (df['count_EV_codes']    > 0) |
        (df['count_Hybrid_codes'] > 0)
    ].copy()
    print(f'  After CPC relevance filter (≥1 ICE/EV/Hybrid code): {len(df):,} rows')

    # Brand mapping
    df['brand'] = df['psn_name'].map(company_map)
    before = len(df)
    df = df.dropna(subset=['brand'])
    df = df[df['brand'].isin(brand_order)].copy()
    print(f'  After brand mapping: {before - len(df):,} rows dropped '
          f'(unmapped PSN_NAMEs), {df["brand"].nunique()} brands retained')

    # Deduplication: one row per appln_id
    # (same patent can appear for multiple subsidiaries of the same brand)
    before = len(df)
    df = df.drop_duplicates(subset=['appln_id']).copy()
    print(f'  After deduplication: {before - len(df):,} duplicate appln_id rows dropped')

    # Technology classification: ICE if ICE codes strictly outnumber EV + Hybrid codes
    # ICE     : count_ICE_codes > count_EV_codes + count_Hybrid_codes
    # EV_Hybrid: count_ICE_codes ≤ count_EV_codes + count_Hybrid_codes
    df['tech_type'] = np.where(
        df['count_ICE_codes'] <= (df['count_EV_codes'] + df['count_Hybrid_codes']),
        'EV_Hybrid', 'ICE'
    )

    df_focal = df[['appln_id', 'docdb_family_id', 'year', 'brand', 'tech_type']].copy()

    # Summary
    print(f'\ndf_focal summary:')
    print(f'  Total rows      : {len(df_focal):,}')
    print(f'  ICE patents     : {(df_focal["tech_type"] == "ICE").sum():,}')
    print(f'  EV_Hybrid patents: {(df_focal["tech_type"] == "EV_Hybrid").sum():,}')
    print(f'  Unique brands   : {df_focal["brand"].nunique()} '
          f'({sorted(df_focal["brand"].unique())})')
    print(f'  Year range      : {df_focal["year"].min()}–{df_focal["year"].max()}')
    return df_focal


df_focal = build_df_focal(US_PATENTS_FILE, YEAR_START, YEAR_END, COMPANY_MAP, BRAND_ORDER)


## 3. FAMSIZE — family size

Count distinct jurisdictions (appln_auth) in each docdb patent family.
Source: `patents_cpc_classified.csv` 


In [ ]:
def extract_famsize(df_focal, cpc_file, chunk_size):
    """
    Compute FAMSIZE: distinct jurisdictions per docdb_family_id.

    Scans patents_cpc_classified.csv in chunks (114 M rows), keeping only
    rows whose docdb_family_id appears in the focal sample.

    Parameters
    ----------
    df_focal   : pd.DataFrame — focal sample
    cpc_file   : Path — patents_cpc_classified.csv (step3 output)
    chunk_size : int — rows per chunk

    Returns
    -------
    pd.DataFrame with columns [docdb_family_id, FAMSIZE]
    """
    focal_families = set(df_focal['docdb_family_id'].unique())
    print(f'Computing FAMSIZE for {len(focal_families):,} unique families...')
    print(f'Reading {cpc_file.name} in chunks of {chunk_size:,} rows...')

    parts = []
    n_chunks = 0
    for chunk in pd.read_csv(
        cpc_file,
        usecols=['docdb_family_id', 'appln_auth'],
        dtype={'docdb_family_id': 'int32', 'appln_auth': 'str'},
        chunksize=chunk_size,
    ):
        sub = chunk[chunk['docdb_family_id'].isin(focal_families)][['docdb_family_id', 'appln_auth']]
        if len(sub) > 0:
            parts.append(sub)
        n_chunks += 1
        if n_chunks % 20 == 0:
            print(f'  ...{n_chunks * chunk_size / 1e6:.0f} M rows processed', end='\r')

    print(f'\n  Done — {n_chunks * chunk_size / 1e6:.0f} M rows scanned')

    all_family_apps = pd.concat(parts, ignore_index=True)
    famsize = (
        all_family_apps
        .groupby('docdb_family_id')['appln_auth']
        .nunique()
        .reset_index(name='FAMSIZE')
    )
    print(f'FAMSIZE results:')
    print(f'  Families found      : {len(famsize):,}')
    print(f'  Median FAMSIZE      : {famsize["FAMSIZE"].median():.1f}')
    print(f'  Mean FAMSIZE        : {famsize["FAMSIZE"].mean():.2f}')
    print(f'  Max FAMSIZE         : {famsize["FAMSIZE"].max()}')
    del all_family_apps, parts
    gc.collect()
    return famsize


# ── FAMSIZE ───────────────────────────────────────────────────────────────────
# Scans ../step3/patents_cpc_classified.csv (~4.6 GB, 114 M rows) and counts the
# distinct patent offices (appln_auth) per docdb_family_id.
famsize_df = extract_famsize(df_focal, CPC_FILE, CHUNK_SIZE)
famsize_df.to_csv(CHECKPOINTS_DIR / 'famsize.csv', index=False)
print(f"Saved: {CHECKPOINTS_DIR / 'famsize.csv'}")

# Families absent from the scan have a single filing -> FAMSIZE = 1
df_focal = df_focal.merge(famsize_df, on='docdb_family_id', how='left')
df_focal['FAMSIZE'] = df_focal['FAMSIZE'].fillna(1).astype(int)
print(f'df_focal after FAMSIZE merge: {len(df_focal):,} rows')


## 4. Citation indicators — BCIT, FCIT, REFNPAT

### Plan
1. **focal_pubs**: scan tls211 → map each focal appln_id to all its pat_publn_ids
2. **Single tls212 scan**: in one pass collect raw data for all three indicators:
   - BCIT raw: rows where a focal pub cites another patent
   - FCIT raw: rows where a non-focal pub cites a focal pub
   - REFNPAT raw: rows where a focal pub cites NPL
3. **Secondary tls211 scan**: resolve appln_ids for cited/citing publications
4. **Compute** BCIT, FCIT, REFNPAT and merge into df_focal

This design minimises tls211 reads (2 passes) and tls212 reads (1 pass).


In [ ]:
def build_focal_pubs(df_focal, tls211_parts, chunk_size):
    """
    Build publication-to-application mapping for all focal applications.

    Scans the PATSTAT table TLS211_PAT_PUBLN — on disk it is the dump split
    into tls211_part01.csv + tls211_part02.csv — to find all publications
    (A1, B1, ...) of each focal application.  A single application can have
    several publications (e.g. the laid-open A1 and the granted patent B1),
    which is why the citation counts below are aggregated back to appln_id.

    Parameters
    ----------
    df_focal      : pd.DataFrame — focal sample with appln_id
    tls211_parts  : list of Path — tls211_part01.csv, tls211_part02.csv
    chunk_size    : int

    Returns
    -------
    pd.DataFrame with columns [appln_id, pat_publn_id]
        One row per publication of a focal application.
    """
    focal_ids = set(df_focal['appln_id'].astype('int32').unique())
    print(f'Building focal_pubs for {len(focal_ids):,} focal appln_ids...')

    parts = []
    for fp in tls211_parts:
        print(f'  Reading {fp.name}...')
        for chunk in pd.read_csv(
            fp,
            usecols=['appln_id', 'pat_publn_id'],
            dtype={'appln_id': 'int32', 'pat_publn_id': 'int32'},
            chunksize=chunk_size,
            on_bad_lines='skip',
        ):
            sub = chunk[chunk['appln_id'].isin(focal_ids)]
            if len(sub) > 0:
                parts.append(sub[['appln_id', 'pat_publn_id']])

    focal_pubs = pd.concat(parts, ignore_index=True).drop_duplicates()
    del parts
    gc.collect()

    print(f'focal_pubs: {len(focal_pubs):,} rows')
    print(f'  Unique focal appln_ids with ≥1 publication : {focal_pubs["appln_id"].nunique():,}')
    print(f'  Unique pat_publn_ids                        : {focal_pubs["pat_publn_id"].nunique():,}')
    apps_no_pub = len(focal_ids) - focal_pubs['appln_id'].nunique()
    print(f'  Focal appln_ids with 0 publications in tls211: {apps_no_pub:,}')
    return focal_pubs


# ── focal_pubs: appln_id -> pat_publn_id ──────────────────────────────────────
# Scans PATSTAT TLS211 (~156 M rows) for the publications of the focal sample.
# One application usually has several publications (laid-open A1 + granted B1),
# so the citation indicators below are aggregated back to the application level.
focal_pubs = build_focal_pubs(df_focal, TLS211_PARTS, CHUNK_SIZE)
focal_pubs.to_csv(CHECKPOINTS_DIR / 'focal_pubs.csv', index=False)
print(f"Saved: {CHECKPOINTS_DIR / 'focal_pubs.csv'}")

focal_pub_ids = set(focal_pubs['pat_publn_id'].unique())
print(f'focal_pub_ids: {len(focal_pub_ids):,} publication ids to search for in tls212')


In [ ]:
def scan_tls212_single_pass(focal_pub_ids, tls212_parts, chunk_size):
    """
    Single-pass scan through the PATSTAT table TLS212_CITATION — on disk
    tls212_part01..03.csv — collecting raw citation data for all three
    indicators simultaneously.

    For each chunk, three non-exclusive filters are applied:
      1. BCIT  — focal pub cites a patent:
                 pat_publn_id ∈ focal_pub_ids AND cited_pat_publn_id > 0
      2. FCIT  — a non-focal pub cites a focal pub:
                 cited_pat_publn_id ∈ focal_pub_ids
      3. REFNPAT — focal pub cites NPL:
                 pat_publn_id ∈ focal_pub_ids AND cited_npl_publn_id > 0

    Parameters
    ----------
    focal_pub_ids : set — from focal_pubs['pat_publn_id']
    tls212_parts  : list of Path
    chunk_size    : int

    Returns
    -------
    tuple (bcit_raw, fcit_raw, refnpat_raw)
        bcit_raw    : DataFrame [pat_publn_id, cited_pat_publn_id]
        fcit_raw    : DataFrame [pat_publn_id, cited_pat_publn_id]
        refnpat_raw : DataFrame [pat_publn_id, cited_npl_publn_id]
    """
    print(f'Scanning tls212 (3 files, single pass) for {len(focal_pub_ids):,} focal pub IDs...')

    bcit_parts, fcit_parts, refnpat_parts = [], [], []

    for fp in tls212_parts:
        print(f'  Reading {fp.name}...')
        n_chunks = 0
        for chunk in pd.read_csv(
            fp,
            usecols=['pat_publn_id', 'cited_pat_publn_id', 'cited_npl_publn_id'],
            dtype={'pat_publn_id'      : 'int32',
                   'cited_pat_publn_id' : 'int32',
                   'cited_npl_publn_id' : 'str'},   # hex hash string in this PATSTAT version
            chunksize=chunk_size,
            on_bad_lines='skip',
        ):
            # BCIT: focal pub cites a patent
            m_bcit = chunk['pat_publn_id'].isin(focal_pub_ids) & (chunk['cited_pat_publn_id'] > 0)
            if m_bcit.any():
                bcit_parts.append(chunk.loc[m_bcit, ['pat_publn_id', 'cited_pat_publn_id']].copy())

            # FCIT: focal pub is cited by a non-focal pub
            m_fcit = chunk['cited_pat_publn_id'].isin(focal_pub_ids)
            if m_fcit.any():
                fcit_parts.append(chunk.loc[m_fcit, ['pat_publn_id', 'cited_pat_publn_id']].copy())

            # REFNPAT: focal pub cites NPL.
            # cited_npl_publn_id is a hex hash when present; sentinel for "no NPL" is '0' or NaN.
            npl_col = chunk['cited_npl_publn_id']
            m_has_npl = npl_col.notna() & (npl_col != '0') & (npl_col != '')
            m_refnpat = chunk['pat_publn_id'].isin(focal_pub_ids) & m_has_npl
            if m_refnpat.any():
                refnpat_parts.append(chunk.loc[m_refnpat, ['pat_publn_id', 'cited_npl_publn_id']].copy())

            n_chunks += 1
            if n_chunks % 50 == 0:
                print(f'    ...{n_chunks * chunk_size / 1e6:.0f} M rows', end='\r')
        print()

    empty_bcit    = pd.DataFrame(columns=['pat_publn_id', 'cited_pat_publn_id'])
    empty_fcit    = pd.DataFrame(columns=['pat_publn_id', 'cited_pat_publn_id'])
    empty_refnpat = pd.DataFrame({'pat_publn_id': pd.Series(dtype='int32'),
                                   'cited_npl_publn_id': pd.Series(dtype='str')})

    bcit_raw    = pd.concat(bcit_parts,    ignore_index=True) if bcit_parts    else empty_bcit
    fcit_raw    = pd.concat(fcit_parts,    ignore_index=True) if fcit_parts    else empty_fcit
    refnpat_raw = pd.concat(refnpat_parts, ignore_index=True) if refnpat_parts else empty_refnpat

    del bcit_parts, fcit_parts, refnpat_parts
    gc.collect()

    print(f'tls212 scan complete:')
    print(f'  BCIT raw rows   : {len(bcit_raw):,}   '
          f'(distinct cited pub IDs: {bcit_raw["cited_pat_publn_id"].nunique():,})')
    print(f'  FCIT raw rows   : {len(fcit_raw):,}   '
          f'(distinct citing pub IDs: {fcit_raw["pat_publn_id"].nunique():,})')
    print(f'  REFNPAT raw rows: {len(refnpat_raw):,}   '
          f'(distinct NPL pub IDs: {refnpat_raw["cited_npl_publn_id"].nunique():,})')
    return bcit_raw, fcit_raw, refnpat_raw


# ── BCIT / FCIT / REFNPAT raw rows from TLS212 ────────────────────────────────
# One pass over PATSTAT TLS212 (~456 M rows), split by the ROLE the focal
# publication plays in each citation row:
#   bcit_raw     focal pub is the CITING side  -> backward citations
#   fcit_raw     focal pub is the CITED side   -> forward citations
#   refnpat_raw  focal pub cites non-patent literature
bcit_raw, fcit_raw, refnpat_raw = scan_tls212_single_pass(
    focal_pub_ids, TLS212_PARTS, CHUNK_SIZE)
bcit_raw.to_csv(CHECKPOINTS_DIR / 'bcit_raw.csv', index=False)
fcit_raw.to_csv(CHECKPOINTS_DIR / 'fcit_raw.csv', index=False)
refnpat_raw.to_csv(CHECKPOINTS_DIR / 'refnpat_raw.csv', index=False)
print(f"Saved: bcit_raw.csv, fcit_raw.csv, refnpat_raw.csv -> {CHECKPOINTS_DIR}")

# Every publication on the OTHER side of a citation: the patents our focal
# patents cite, and the patents that cite them.
secondary_pub_ids = (set(bcit_raw['cited_pat_publn_id'].unique())
                     | set(fcit_raw['pat_publn_id'].unique()))
print(f'secondary_pub_ids: {len(secondary_pub_ids):,} publication ids to resolve back to applications')


In [ ]:
def resolve_publn_to_appln(pub_id_set, tls211_parts, chunk_size, label=''):
    """
    Resolve a set of pat_publn_ids to their appln_ids via TLS211_PAT_PUBLN
    (tls211_part01.csv + tls211_part02.csv).

    Parameters
    ----------
    pub_id_set   : set of int — pat_publn_ids to resolve
    tls211_parts : list of Path
    chunk_size   : int
    label        : str — for progress messages

    Returns
    -------
    pd.DataFrame with columns [pat_publn_id, appln_id]
        Deduplicated to one appln_id per pat_publn_id.
    """
    print(f'Resolving {len(pub_id_set):,} {label} pub_ids to appln_ids...')
    parts = []
    for fp in tls211_parts:
        for chunk in pd.read_csv(
            fp,
            usecols=['pat_publn_id', 'appln_id'],
            dtype={'pat_publn_id': 'int32', 'appln_id': 'int32'},
            chunksize=chunk_size,
            on_bad_lines='skip',
        ):
            sub = chunk[chunk['pat_publn_id'].isin(pub_id_set)]
            if len(sub) > 0:
                parts.append(sub[['pat_publn_id', 'appln_id']])

    result = (pd.concat(parts, ignore_index=True) if parts
              else pd.DataFrame(columns=['pat_publn_id', 'appln_id']))
    # Keep first appln_id if multiple exist (rare edge case)
    result = result.drop_duplicates(subset=['pat_publn_id'])
    print(f'  Resolved {len(result):,} pub_id → appln_id mappings '
          f'(coverage: {len(result) / max(len(pub_id_set), 1) * 100:.1f}%)')
    return result


# ── secondary_map: pat_publn_id -> appln_id ───────────────────────────────────
# Second pass over TLS211. TLS212 links PUBLICATIONS, but the indicators must be
# counted per APPLICATION - otherwise a patent citing both the A1 and the B1
# publication of the same prior filing would be counted twice.
secondary_map = resolve_publn_to_appln(
    secondary_pub_ids, TLS211_PARTS, CHUNK_SIZE, 'secondary')
secondary_map.to_csv(CHECKPOINTS_DIR / 'secondary_map.csv', index=False)
print(f"Saved: {CHECKPOINTS_DIR / 'secondary_map.csv'}")


In [ ]:
# ── BCIT — backward citations ─────────────────────────────────────────────────

def compute_bcit(bcit_raw, focal_pubs, secondary_map):
    """
    Count distinct cited applications per focal application.

    Join chain:
      bcit_raw [pat_publn_id → cited_pat_publn_id]
      ↑ focal_pubs     [pat_publn_id  → appln_id (focal)]
      ↓ secondary_map  [cited_pat_publn_id → cited_appln_id]
    Group by focal appln_id, count nunique(cited_appln_id).

    Parameters
    ----------
    bcit_raw      : DataFrame [pat_publn_id, cited_pat_publn_id]
    focal_pubs    : DataFrame [appln_id, pat_publn_id]
    secondary_map : DataFrame [pat_publn_id, appln_id]

    Returns
    -------
    pd.DataFrame with columns [appln_id, BCIT]
    """
    # Map focal pub → focal appln_id
    m1 = bcit_raw.merge(
        focal_pubs[['appln_id', 'pat_publn_id']],
        on='pat_publn_id',
        how='inner'
    )
    # Map cited pub → cited appln_id
    m2 = m1.merge(
        secondary_map.rename(columns={
            'pat_publn_id': 'cited_pat_publn_id',
            'appln_id'    : 'cited_appln_id',
        }),
        on='cited_pat_publn_id',
        how='inner'
    )
    bcit = (
        m2.groupby('appln_id')['cited_appln_id']
        .nunique()
        .reset_index(name='BCIT')
    )
    print(f'BCIT: {len(bcit):,} focal apps with ≥1 backward citation')
    print(f'  Mean  : {bcit["BCIT"].mean():.2f}')
    print(f'  Median: {bcit["BCIT"].median():.1f}')
    return bcit


# ── FCIT — forward citations ──────────────────────────────────────────────────

def compute_fcit(fcit_raw, focal_pubs, secondary_map):
    """
    Count distinct citing applications per focal application.

    fcit_raw columns: [pat_publn_id (citing), cited_pat_publn_id (focal)]

    Join chain:
      fcit_raw [cited_pat_publn_id → pat_publn_id (citing)]
      ↑ focal_pubs     [pat_publn_id(focal)  → appln_id (focal)]
      ↓ secondary_map  [pat_publn_id(citing) → citing_appln_id]
    Group by focal appln_id, count nunique(citing_appln_id).

    Parameters
    ----------
    fcit_raw      : DataFrame [pat_publn_id, cited_pat_publn_id]
    focal_pubs    : DataFrame [appln_id, pat_publn_id]
    secondary_map : DataFrame [pat_publn_id, appln_id]

    Returns
    -------
    pd.DataFrame with columns [appln_id, FCIT]
    """
    # Map cited (focal) pub → focal appln_id
    m1 = fcit_raw.merge(
        focal_pubs[['appln_id', 'pat_publn_id']].rename(
            columns={'pat_publn_id': 'cited_pat_publn_id'}
        ),
        on='cited_pat_publn_id',
        how='inner'
    )
    # Map citing pub → citing appln_id
    m2 = m1.merge(
        secondary_map.rename(columns={'appln_id': 'citing_appln_id'}),
        on='pat_publn_id',
        how='inner'
    )
    fcit = (
        m2.groupby('appln_id')['citing_appln_id']
        .nunique()
        .reset_index(name='FCIT')
    )
    print(f'FCIT: {len(fcit):,} focal apps with ≥1 forward citation')
    print(f'  Mean  : {fcit["FCIT"].mean():.2f}')
    print(f'  Median: {fcit["FCIT"].median():.1f}')
    return fcit


# ── REFNPAT — non-patent-literature references ────────────────────────────────

def compute_refnpat(refnpat_raw, focal_pubs):
    """
    Count distinct NPL documents cited per focal application.

    No secondary tls211 lookup needed: cited_npl_publn_id is the NPL document
    identifier directly from tls212.  We count distinct NPL IDs per focal app.

    Parameters
    ----------
    refnpat_raw : DataFrame [pat_publn_id, cited_npl_publn_id]
    focal_pubs  : DataFrame [appln_id, pat_publn_id]

    Returns
    -------
    pd.DataFrame with columns [appln_id, REFNPAT]
    """
    merged = refnpat_raw.merge(
        focal_pubs[['appln_id', 'pat_publn_id']],
        on='pat_publn_id',
        how='inner'
    )
    refnpat = (
        merged.groupby('appln_id')['cited_npl_publn_id']
        .nunique()
        .reset_index(name='REFNPAT')
    )
    print(f'REFNPAT: {len(refnpat):,} focal apps with ≥1 NPL citation')
    print(f'  Mean  : {refnpat["REFNPAT"].mean():.2f}')
    print(f'  Median: {refnpat["REFNPAT"].median():.1f}')
    return refnpat


# Compute all three
bcit_df    = compute_bcit(bcit_raw, focal_pubs, secondary_map)
fcit_df    = compute_fcit(fcit_raw, focal_pubs, secondary_map)
refnpat_df = compute_refnpat(refnpat_raw, focal_pubs)

# Merge into df_focal
df_focal = df_focal.merge(bcit_df,    on='appln_id', how='left')
df_focal = df_focal.merge(fcit_df,    on='appln_id', how='left')
df_focal = df_focal.merge(refnpat_df, on='appln_id', how='left')
df_focal[['BCIT', 'FCIT', 'REFNPAT']] = (
    df_focal[['BCIT', 'FCIT', 'REFNPAT']].fillna(0).astype(int)
)

print(f'\ndf_focal with all 4 indicators: {len(df_focal):,} rows')
print(df_focal[['FAMSIZE', 'BCIT', 'FCIT', 'REFNPAT']].describe().round(2))

# Save the patent-level indicator table.
# Consumed by: table2_descriptive_stats.py (Table 2),
#              indicator_trends.py (Figure 3),
#              indicator_trends_famsize_german.py (Figure 4),
#              ../04_ddd_regressions/01_build_panel.py (Tables 3-5, Appendix A5),
#              ../03_patent_count_figures/build_figure1_and_appendixA3.py (Figure 1, Appendix A3)
OUT_CP = CHECKPOINTS_DIR / 'df_focal_indicators.csv'
df_focal.to_csv(OUT_CP, index=False)
print(f'\nSaved: {OUT_CP}')


## 5. Density plots

Raw-value density plot per indicator → `output/plots/raw_density/`.

`FAMSIZE_raw_density.png` is Figure 2; the BCIT / FCIT / REFNPAT plots are
Appendix A4.1–A4.3. Table 2 (descriptive statistics) is produced separately
by `table2_descriptive_stats.py`.


In [ ]:
INDICATORS = ['FAMSIZE', 'BCIT', 'FCIT', 'REFNPAT']
INDICATOR_LABELS = {
    'FAMSIZE' : 'Family Size (jurisdictions)',
    'BCIT'    : 'Backward Citations (patent)',
    'FCIT'    : 'Forward Citations (patent)',
    'REFNPAT' : 'NPL References',
}

print(df_focal[INDICATORS].describe().round(2))


In [ ]:
# ── Raw density plots ─────────────────────────────────────────────────────────

def plot_raw_density(df, indicators, labels, output_dir):
    """
    One plot per indicator showing the raw-value density with annotations.
    Annotates: share of zeros, mean, median.
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    for ind in indicators:
        s = df[ind]

        fig, ax = plt.subplots(figsize=(8, 4))

        # Clip extreme values for display (keep the shape visible)
        p99 = s.quantile(0.99)
        s_clip = s.clip(upper=p99)

        ax.hist(s_clip, bins=60, density=True, color='steelblue', edgecolor='none', alpha=0.8)

        ax.set_xlabel(f'{labels[ind]}', fontsize=11)
        ax.set_ylabel('Density', fontsize=11)
        ax.grid(True, linestyle=':', alpha=0.4)
        plt.tight_layout()

        out = output_dir / f'{ind}_raw_density.png'
        fig.savefig(out, dpi=200, bbox_inches='tight')
        plt.close(fig)
        print(f'  Saved: {out.name}')


plot_raw_density(df_focal[df_focal['year'].between(2010, 2019)], INDICATORS, INDICATOR_LABELS, PLOTS_RAW)
